# 配体库提取 + 双 Fe 配合物组合生成

## 从 108k tmQM 配合物中提取配体，重组生成新候选分子

### 流程
```
tmQM_y.csv (108k) → 识别金属中心 → 切断M-L键 → 提取配体SMILES
                                            ↓
                       按配位原子(N/O/P/S/Cl...)分类
                                            ↓
                       选Fe相关配体 → 组合到双Fe骨架上
                                            ↓
                       输出候选SMILES → GNN筛选
```


In [ ]:
# ============================================================
# Step 1: 加载数据
# ============================================================
import numpy as np, pandas as pd
from collections import Counter, defaultdict
from rdkit import Chem
from rdkit.Chem import AllChem, Draw, rdmolops
from rdkit import RDLogger; RDLogger.logger().setLevel(RDLogger.ERROR)
import warnings; warnings.filterwarnings('ignore')

df = pd.read_csv('../chem_ml5/tmQM_y.csv', sep=';')
df = df.rename(columns={
    'HL_Gap': 'gap', 'HOMO_Energy': 'homo', 'LUMO_Energy': 'lumo',
    'Dipole_M': 'dipole', 'Metal_q': 'metal_charge',
    'Electronic_E': 'E_elec'
})
print(f'Loaded {len(df):,} complexes')
df[['CSD_code', 'SMILES']].head()


In [ ]:
# ============================================================
# Step 2: 识别金属中心 + 切断 M-L 键提取配体
# ============================================================

# 过渡金属原子序数
TM_SET = set([21,22,23,24,25,26,27,28,29,30,39,40,41,42,43,44,45,46,47,48,
             72,73,74,75,76,77,78,79,80])
# 常见配位原子
DONOR_SET = {7, 8, 15, 16, 17, 35, 53}  # N, O, P, S, Cl, Br, I


def extract_ligands(smiles):
    """从配合物SMILES中提取配体片段"""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None

    # 找到金属原子
    metal_idxs = [a.GetIdx() for a in mol.GetAtoms() if a.GetAtomicNum() in TM_SET]
    if not metal_idxs: return None

    # 找到所有 M-L 键
    ml_bonds = []
    for bond in mol.GetBonds():
        a1, a2 = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        z1 = mol.GetAtomWithIdx(a1).GetAtomicNum()
        z2 = mol.GetAtomWithIdx(a2).GetAtomicNum()
        is_metal_1 = z1 in TM_SET
        is_metal_2 = z2 in TM_SET
        if is_metal_1 != is_metal_2:  # 恰好一个是金属
            donor_idx = a2 if is_metal_1 else a1
            donor_z = z2 if is_metal_1 else z1
            if donor_z in DONOR_SET:
                ml_bonds.append(bond.GetIdx())

    if not ml_bonds: return None

    # 切断 M-L 键，提取碎片
    frag_mol = Chem.FragmentOnBonds(mol, ml_bonds, addDummies=True)
    if frag_mol is None: return None

    # 获取碎片 SMILES
    frag_smiles_list = Chem.MolToSmiles(frag_mol).split('.')

    # 分离金属碎片和配体碎片
    ligands = []
    metal_frag = None
    for fs in frag_smiles_list:
        f_mol = Chem.MolFromSmiles(fs)
        if f_mol is None: continue
        # 检查是否含金属
        has_metal = any(a.GetAtomicNum() in TM_SET for a in f_mol.GetAtoms())
        has_dummy = any(a.GetAtomicNum() == 0 for a in f_mol.GetAtoms())

        if has_metal:
            metal_frag = fs
        elif has_dummy:
            # 配体片段：去掉dummy原子后的纯净SMILES
            ligands.append(fs)

    return {
        'metal_frag': metal_frag,
        'ligands': ligands,
        'n_donors': len(ml_bonds),
        'metal_types': [mol.GetAtomWithIdx(i).GetSymbol() for i in metal_idxs]
    }


# 测试
test_smi = df['SMILES'].values[10]
result = extract_ligands(test_smi)
if result:
    print(f'SMILES: {test_smi[:60]}...')
    print(f'Metal: {result["metal_types"]}')
    print(f'Ligands ({len(result["ligands"])}):')
    for lig in result['ligands'][:5]:
        print(f'  {lig}')
    print(f'Total M-L bonds: {result["n_donors"]}')


In [ ]:
# ============================================================
# Step 3: 批量提取，建立配体库
# ============================================================
from tqdm import tqdm

# 筛选Fe配合物作为提取源
fe_mask = df['SMILES'].str.contains('Fe', na=False)
df_fe = df[fe_mask].head(3000)  # 用3000个Fe配合物提取配体
print(f'Processing {len(df_fe)} Fe complexes...')

ligand_db = defaultdict(list)  # {donor_element: [(smiles, count), ...]}
metal_scaffolds = []  # Fe骨架
stats = {'success': 0, 'failed': 0}

for _, row in tqdm(df_fe.iterrows(), total=len(df_fe)):
    result = extract_ligands(row['SMILES'])
    if result and result['ligands']:
        stats['success'] += 1
        if result['metal_frag']:
            metal_scaffolds.append(result['metal_frag'])
        for lig in result['ligands']:
            # 识别配位原子
            lmol = Chem.MolFromSmiles(lig)
            if lmol is None: continue
            for a in lmol.GetAtoms():
                if a.GetAtomicNum() == 0:  # dummy原子 = 配位点
                    for nb in a.GetNeighbors():
                        donor_z = nb.GetAtomicNum()
                        if donor_z in DONOR_SET:
                            donor_sym = nb.GetSymbol()
                            # 存去dummy的纯净配体
                            clean_mol = Chem.DeleteSubstructs(
                                lmol, Chem.MolFromSmarts('[#0]'))
                            if clean_mol:
                                clean_smi = Chem.MolToSmiles(clean_mol)
                                ligand_db[donor_sym].append(clean_smi)
    else:
        stats['failed'] += 1

print(f'\nStats: {stats}')
print(f'Unique Fe scaffolds: {len(set(metal_scaffolds))}')
print(f'\nLigand library:')
for donor, ligs in sorted(ligand_db.items()):
    unique = set(ligs)
    print(f'  {donor}-donor: {len(ligs)} total, {len(unique)} unique')


In [ ]:
# ============================================================
# Step 4: 去重 + 筛选常见配体
# ============================================================

ligand_unique = {}
MIN_FREQ = 3  # 至少出现3次
for donor, ligs in ligand_db.items():
    counter = Counter(ligs)
    filtered = {smi: cnt for smi, cnt in counter.items() if cnt >= MIN_FREQ}
    ligand_unique[donor] = dict(sorted(filtered.items(), key=lambda x: -x[1])[:200])

for donor, ligs in ligand_unique.items():
    print(f'{donor}: {len(ligs)} unique ligands (freq >= {MIN_FREQ})')
    if ligs:
        top3 = list(ligs.items())[:3]
        for smi, cnt in top3:
            print(f'  [{cnt}x] {smi[:60]}')

total_ligands = sum(len(v) for v in ligand_unique.values())
print(f'\nTotal ligand pool: {total_ligands}')


In [ ]:
# ============================================================
# Step 5: 定义 Fe 骨架 + 组合生成
# ============================================================

# 定义几个常见 Fe 配位几何骨架
# [Fe] = dummy原子标记Fe位置，数字=配位数
scaffolds = {
    'Fe_octa_6':  '[Fe]([L1])([L2])([L3])([L4])([L5])[L6]',   # 6配位八面体
    'Fe_sqplan_4': '[Fe]([L1])([L2])([L3])[L4]',               # 4配位平面四方
    'Fe_trig_5':   '[Fe]([L1])([L2])([L3])([L4])[L5]',         # 5配位
}

# 选配体：从每类donor取top-N
N_PER_DONOR = 10
selected_ligands = {}
for donor, ligs in ligand_unique.items():
    selected_ligands[donor] = list(ligs.keys())[:N_PER_DONOR]

for donor, ligs in selected_ligands.items():
    print(f'{donor}: {len(ligs)} ligands selected')

# 组合生成（简化版：随机采样）
import random
random.seed(42)

print('\nGenerating Fe(octa_6) complexes with mixed ligands...')
generated = []
for _ in range(1000):  # 生成1000个
    # 随机选配位原子类型
    donor_types = random.choices(list(selected_ligands.keys()), k=6)
    ligands_chosen = []
    for dt in donor_types:
        if selected_ligands[dt]:
            ligands_chosen.append(random.choice(selected_ligands[dt]))

    if len(ligands_chosen) == 6:
        # 组装SMILES: [Fe](L1)(L2)...(L6)
        parts = ['[Fe]'] + [f'({l})' for l in ligands_chosen]
        gen_smi = ''.join(parts)
        generated.append(gen_smi)

print(f'Generated {len(generated)} candidate complexes')
print('\nExamples:')
for smi in generated[:5]:
    print(f'  {smi}')


## 下一步

生成的候选 SMILES 列表可以：
1. 喂给 tmQM_Fe.ipynb 的 GAT GNN 预测 8 个 DFT 性质
2. 按 gap/dipole/metal_charge 排序筛选
3. top-10 跑 DFT (ORCA) 精确验证
4. 提交实验

---

**当前状态：**
- 阶段一(基准模型)：已完成 ✅
- 阶段二(配体库+组合生成)：本 notebook ✅
- 阶段三(GNN筛选)：待算力就绪/替换为RF版
- 阶段四(DFT验证+实验)：待推进
